# SLM-SIP: MCQ Challenge — Llama-3.2-1B Post-Training

Fine-tuning pipeline for the Smart MCQ Solver Challenge using Hugging Face Transformers + PEFT (LoRA) on Colab GPU.

**Runtime**: Runtime > Change runtime type > T4 GPU (or A100 if available).

**Data**: Upload `train.csv`, `test.csv`, `sample_submission.csv` to the Colab working dir (or mount Drive and adjust `DATA_DIR` below).

In [ ]:
# Install fine-tuning stack (Colab ships torch already)
!pip install -q -U transformers==4.46.3 peft==0.13.2 datasets==3.1.0 accelerate==1.1.1 bitsandbytes==0.44.1 trl==0.12.1 scikit-learn==1.5.2

In [ ]:
import os, json, torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

assert torch.cuda.is_available(), "Enable GPU: Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda, "| torch:", torch.__version__)

In [ ]:
# --- Config ---
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"   # gated; ensure HF token is set below
HF_TOKEN = os.environ.get("HF_TOKEN", "")        # or paste token; Colab Secrets recommended
assert HF_TOKEN, "Set HF_TOKEN (meta-llama is gated): Colab > Secrets > add HF_TOKEN"
from huggingface_hub import login
login(token=HF_TOKEN)

# Data
DATA_DIR = "."   # Colab working dir after upload; change if using Drive
TRAIN_CSV = f"{DATA_DIR}/train.csv"
TEST_CSV  = f"{DATA_DIR}/test.csv"
SAMPLE_SUB = f"{DATA_DIR}/sample_submission.csv"

# Training
MAX_LEN = 512
BATCH = 4
GRAD_ACCUM = 4
LR = 2e-4
EPOCHS = 3
USE_4BIT = True
OUTPUT_DIR = "./llama1b-mcq-lora"

In [ ]:
# --- Load data ---
train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)
sub   = pd.read_csv(SAMPLE_SUB)
print("train:", train.shape, "| test:", test.shape)
print(train.columns.tolist())
print("\n--- sample row ---")
print(train.iloc[0].to_dict())

In [ ]:
# --- Tokenizer + model (4-bit QLoRA-ready) ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

quant_cfg = BitsAndBytesConfig(
    load_in_4bit=USE_4BIT,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=quant_cfg if USE_4BIT else None,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

In [ ]:
# --- LoRA adapter ---
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## Prompt formatting + dataset prep

TODO (yours to drive): inspect the column schema from the sample row above and build the instruction-style prompt template + label masking for the MCQ task. HuggingFace `datasets.Dataset.from_pandas(...)` after you've constructed the `text` column.

In [ ]:
# TODO: build prompts from train/test columns here
# e.g. question + options A/B/C/D -> target answer letter
# from datasets import Dataset
# ds = Dataset.from_pandas(train_df_with_text)
# tokenized = ds.map(lambda x: tokenizer(x["text"], truncation=True, max_length=MAX_LEN), batched=True)

## Training

TODO (yours to drive): plug into `SFTTrainer` (trl) with the `TrainingArguments` below, or roll your own loop. Key knobs to revisit: LR, epochs, `MAX_LEN`, LoRA `r`.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=10,
    save_steps=200,
    bf16=True,
    report_to="none",
    remove_unused_columns=False,
)

# from trl import SFTTrainer
# trainer = SFTTrainer(model=model, args=training_args, train_dataset=tokenized, ...)
# trainer.train()
# trainer.save_model(OUTPUT_DIR)

In [ ]:
# --- Inference / submission stub ---
# TODO: generate on test set with the same prompt template, extract predicted answer
# to match sample_submission.csv columns, then write submission.csv
print(sub.columns.tolist())